# Who Wins the 2026 World Cup? A 10,000-Simulation Forecast

Companion notebook to ***Soccer Analytics with Machine Learning*** (O'Reilly, 2026).

We rate every team with **World Football Elo**, turn each matchup into a **Poisson goal model**
(the approach from Chapters 4 and 6), and simulate the entire tournament 10,000 times to get each
team's probability of lifting the trophy.

> **Before you publish:** the Elo values in the `ELO` dict are an illustrative early-2026 snapshot.
> Replace them with live numbers from [eloratings.net](https://www.eloratings.net/) on the morning
> you post, then re-run. Nothing else needs to change.


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)  # reproducible

## 2. The field: confirmed 2026 draw + an Elo snapshot

In [ ]:
GROUPS = {
    "A": ["Mexico", "South Africa", "South Korea", "Czechia"],
    "B": ["Canada", "Bosnia-Herzegovina", "Qatar", "Switzerland"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "T\u00fcrkiye"],
    "E": ["Germany", "Curacao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "Congo DR", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

# Illustrative Elo snapshot — REFRESH BEFORE PUBLISHING
ELO = {
    "Spain": 2165, "Argentina": 2120, "France": 2100, "England": 2055,
    "Brazil": 2025, "Netherlands": 2030, "Portugal": 2010, "Germany": 1965,
    "Belgium": 1950, "Croatia": 1945, "Uruguay": 1930, "Colombia": 1915,
    "Morocco": 1900, "Japan": 1900, "Senegal": 1895, "Switzerland": 1860,
    "Norway": 1855, "Austria": 1850, "Ecuador": 1840, "T\u00fcrkiye": 1840,
    "Mexico": 1820, "Czechia": 1815, "Sweden": 1815, "United States": 1805,
    "Iran": 1800, "Ivory Coast": 1800, "Algeria": 1795, "South Korea": 1790,
    "Scotland": 1780, "Egypt": 1780, "Canada": 1780, "Ghana": 1750,
    "Paraguay": 1720, "Australia": 1720, "Congo DR": 1720, "Bosnia-Herzegovina": 1710,
    "Tunisia": 1700, "Qatar": 1680, "Uzbekistan": 1680, "Saudi Arabia": 1655,
    "Iraq": 1650, "Panama": 1650, "South Africa": 1640, "Jordan": 1600,
    "Cape Verde": 1555, "Curacao": 1530, "Haiti": 1500, "New Zealand": 1500,
}

## 3. From Elo to goals

A 400-point Elo edge is worth roughly one goal of supremacy. We split a baseline of 2.7 total
goals between the two sides according to their Elo gap, then draw each team's goals from a
Poisson distribution — exactly the count model the book builds in Chapter 6.

In [ ]:
GOALS_BASE = 2.7
GOALS_PER_400_ELO = 1.0

def lambdas(a, b):
    diff = (ELO[a] - ELO[b]) / 400.0 * GOALS_PER_400_ELO
    la = max(0.15, GOALS_BASE / 2 + diff / 2)
    lb = max(0.15, GOALS_BASE / 2 - diff / 2)
    return la, lb

def play(a, b, knockout=False):
    la, lb = lambdas(a, b)
    ga, gb = rng.poisson(la), rng.poisson(lb)
    if ga != gb:
        return (a, ga, gb) if ga > gb else (b, ga, gb)
    if not knockout:
        return (None, ga, gb)
    p = min(0.75, max(0.25, 0.5 + (ELO[a] - ELO[b]) / 4000.0))  # penalties
    return (a if rng.random() < p else b, ga, gb)

## 4. Group stage, then a seeded knockout bracket

In [ ]:
def run_group(teams):
    pts = {t: 0 for t in teams}; gd = {t: 0 for t in teams}; gf = {t: 0 for t in teams}
    for i in range(len(teams)):
        for j in range(i + 1, len(teams)):
            a, b = teams[i], teams[j]
            w, ga, gb = play(a, b)
            gf[a] += ga; gf[b] += gb; gd[a] += ga - gb; gd[b] += gb - ga
            if w is None: pts[a] += 1; pts[b] += 1
            else: pts[w] += 3
    order = sorted(teams, key=lambda t: (pts[t], gd[t], gf[t], rng.random()), reverse=True)
    rows = [{"team": t, "pts": pts[t], "gd": gd[t], "gf": gf[t]} for t in order]
    return order, rows

def bracket_seed_order(n):
    seeds = [1, 2]
    while len(seeds) < n:
        m = len(seeds) * 2 + 1
        seeds = [s for x in seeds for s in (x, m - x)]
    return seeds

def run_tournament():
    winners, runners, thirds = [], [], []
    for g, teams in GROUPS.items():
        order, rows = run_group(teams)
        winners.append(order[0]); runners.append(order[1])
        thirds.append({"team": order[2], **{k: rows[2][k] for k in ("pts","gd","gf")}})
    thirds.sort(key=lambda r: (r["pts"], r["gd"], r["gf"], rng.random()), reverse=True)
    best_thirds = [r["team"] for r in thirds[:8]]
    winners.sort(key=lambda t: ELO[t], reverse=True)
    runners.sort(key=lambda t: ELO[t], reverse=True)
    best_thirds.sort(key=lambda t: ELO[t], reverse=True)
    seeded = winners + runners + best_thirds
    bracket = [seeded[s - 1] for s in bracket_seed_order(32)]
    rounds = ["R32", "R16", "QF", "SF", "Final", "Champion"]
    reached = {t: "R32" for t in bracket}
    field = bracket
    for ridx in range(5):
        nxt = []
        for i in range(0, len(field), 2):
            w, _, _ = play(field[i], field[i + 1], knockout=True)
            nxt.append(w); reached[w] = rounds[ridx + 1]
        field = nxt
    return field[0], reached

# NOTE: this uses a seeded bracket (group winners kept apart). To use FIFA's exact
# Round-of-32 map, plug the official pairings into `bracket` — the sim logic is identical.

## 5. Simulate 10,000 tournaments

In [ ]:
N = 10000
teams = list(ELO.keys())
title = {t: 0 for t in teams}; finalist = {t: 0 for t in teams}; semi = {t: 0 for t in teams}
for _ in range(N):
    champ, reached = run_tournament()
    title[champ] += 1
    for t, r in reached.items():
        if r in ("Final", "Champion"): finalist[t] += 1
        if r in ("SF", "Final", "Champion"): semi[t] += 1

df = (pd.DataFrame({
        "team": teams,
        "title_pct": [100 * title[t] / N for t in teams],
        "final_pct": [100 * finalist[t] / N for t in teams],
        "semi_pct":  [100 * semi[t] / N for t in teams],
        "elo": [ELO[t] for t in teams],
    }).sort_values("title_pct", ascending=False).reset_index(drop=True))
df.head(15)

## 6. The headline chart

In [ ]:
top = df.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top["team"], top["title_pct"], color="#1E2761")
bars[-1].set_color("#B85042")
for b, v in zip(bars, top["title_pct"]):
    ax.text(v + 0.2, b.get_y() + b.get_height()/2, f"{v:.1f}%", va="center")
ax.set_xlabel("Probability of winning the 2026 World Cup")
ax.set_title("Who wins the 2026 World Cup?\n10,000 simulations \u2014 an Elo + Poisson model",
             fontweight="bold", loc="left")
ax.spines[["top", "right"]].set_visible(False)
ax.set_xlim(0, top["title_pct"].max() * 1.18)
plt.tight_layout()
plt.savefig("wc2026_title_probabilities.png", dpi=160, bbox_inches="tight")
plt.show()

## What this shows — and what it leaves out

The favorites that fall out of the model are the ones you'd expect, which is the point: a few
dozen lines of Elo + Poisson recover the same picture as the betting market. The book goes
further — calibrating the goal model on real match data, adding home advantage and squad
strength, and (Chapter 9) comparing these probabilities to bookmaker odds to find value.

*From* **Soccer Analytics with Machine Learning** *(O'Reilly, 2026) by Haipeng Gao, Ari Joury,
Weining Shen, and Guanyu Hu.*